<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Practical: Data Wrangling with PySpark (EU Sales)

*Session 5 · Notebook 01.03 · Practical · Coach version*

## About this notebook

This practical puts the PySpark walkthrough (notebook 01.02, Part 2) to work: reading several files, quick checks, aggregating, chaining joins across five tables, filtering, ranking, writing SQL, and saving a result. The business question is simple: **which EU countries generate the most revenue for the Pagila DVD-rental business?** Answering it needs data from five tables that only make sense together.

It is the PySpark twin of the Session 2 pandas practical on the same data: same question, same tables, new tool. The same code would work on billions of payments.

## About the exercises

Each task appears as three cells:

1. A markdown cell describing what to do (sometimes with a `> *Hint:*`).
2. A `# Your turn` cell for you to write your answer.
3. A `# Coach answer` cell with worked, runnable code; this is the coach version, so every answer is filled in and the notebook runs top-to-bottom with `Run All`.

Work top to bottom: later tasks reuse variables from earlier answers.

## About the data

Five tables from the **Pagila** DVD-rental database (the same data as Session 2), linked by id columns:

```
payment --customer_id--> customer --address_id--> address --city_id--> city --country_id--> country
```

| Table | Key columns | Useful columns |
|---|---|---|
| `payment` (7 monthly files, Jan to Jul 2022) | `payment_id`, `customer_id` | `amount`, `payment_date` |
| `customer` | `customer_id`, `address_id` | `first_name`, `last_name`, `email`, `active` |
| `address` | `address_id`, `city_id` | `address`, `district` |
| `city` | `city_id`, `country_id` | `city` |
| `country` | `country_id` | `country` |

Most tables also carry a `last_update` timestamp. It is not needed for the analysis, and because it appears in several tables it will cause **duplicate column names** if you join whole tables. Select only the columns you need before each join.

## Index

0. [Setup: Spark and the data](#setup)
1. [Explore and check the data](#explore)
2. [Revenue per customer](#customer)
3. [Join the location tables](#joins)
4. [Revenue per country and the EU filter](#eu)
5. [The same answer in SQL](#sql)
6. [Visualise and save](#save)
7. [Capstone](#capstone)
8. [Further practice](#further)

<a id="setup"></a>
## 0. Setup: Spark and the data

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # finds config.py next to this notebook or one level up

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from config import pandas_bridge, session_datasets, session_datasets_http

# On Databricks this returns the session that already exists; on a laptop it starts one.
spark = SparkSession.builder.appName("05_01_03_Practical_EUSales").getOrCreate()
print("Spark version:", spark.version)

The five tables are loaded for you. Note the payment table: it is stored as seven monthly files. The pandas bridge downloads all seven and stacks them; the Spark alternative reads them in one go with a wildcard path (`payment_p2022_0*.csv`).

In [ ]:
# Default: the pandas bridge. pandas downloads the file from the course S3 bucket over HTTPS and
# Spark converts the result. Works on every compute; fine for training-size files, never for
# real big data (the whole file passes through the driver's memory).
payment = pandas_bridge(spark, "pagila_payment")

# Alternative 1: the same two steps in plain code. pandas reads the file, Spark converts it.
# (The bridge also turns pandas' missing values, NaN, into proper Spark nulls; this version does not.)
# pdf = pd.concat([pd.read_csv(url) for url in session_datasets_http["pagila_payment"]], ignore_index=True)
# payment = spark.createDataFrame(pdf)

# Alternative 2: Spark reads the file straight from S3 (the big-data way; needs S3 access on your compute)
# payment = spark.read.csv(session_datasets["pagila_payment"], header=True, inferSchema=True)

In [ ]:
# Default: the pandas bridge. pandas downloads the file from the course S3 bucket over HTTPS and
# Spark converts the result. Works on every compute; fine for training-size files, never for
# real big data (the whole file passes through the driver's memory).
customer = pandas_bridge(spark, "pagila_customer")

# Alternative 1: the same two steps in plain code. pandas reads the file, Spark converts it.
# (The bridge also turns pandas' missing values, NaN, into proper Spark nulls; this version does not.)
# pdf = pd.read_csv(session_datasets_http["pagila_customer"])
# customer = spark.createDataFrame(pdf)

# Alternative 2: Spark reads the file straight from S3 (the big-data way; needs S3 access on your compute)
# customer = spark.read.csv(session_datasets["pagila_customer"], header=True, inferSchema=True)

In [ ]:
# The three location tables, read the same way
address = pandas_bridge(spark, "pagila_address")
city    = pandas_bridge(spark, "pagila_city")
country = pandas_bridge(spark, "pagila_country")

# Alternative (plain pandas, then Spark):
# address = spark.createDataFrame(pd.read_csv(session_datasets_http["pagila_address"]))
# city    = spark.createDataFrame(pd.read_csv(session_datasets_http["pagila_city"]))
# country = spark.createDataFrame(pd.read_csv(session_datasets_http["pagila_country"]))

# Alternative (Spark reads S3):
# address = spark.read.csv(session_datasets["pagila_address"], header=True, inferSchema=True)
# city    = spark.read.csv(session_datasets["pagila_city"], header=True, inferSchema=True)
# country = spark.read.csv(session_datasets["pagila_country"], header=True, inferSchema=True)

In [ ]:
EU_COUNTRIES = ["Austria", "Belgium", "Bulgaria", "Croatia", "Czech Republic", "Denmark", "Estonia",
                "Finland", "France", "Germany", "Greece", "Hungary", "Ireland", "Italy", "Latvia", "Lithuania",
                "Luxembourg", "Malta", "Netherlands", "Poland", "Portugal", "Romania", "Slovakia", "Slovenia",
                "Spain", "Sweden"]

<a id="explore"></a>
## 1. Explore and check the data

### Exercise 1: Size and schema

Print the number of rows and the schema of each of the five tables.

> *Hint:* Loop over a list of `(name, DataFrame)` pairs.

In [ ]:
# Your turn

In [ ]:
# Coach answer
for name, df in [("payment", payment), ("customer", customer), ("address", address), ("city", city), ("country", country)]:
    print(f"{name}: {df.count():,} rows")
    df.printSchema()

### Exercise 2: Payment quality checks

Check the `payment` table: count the nulls in `customer_id` and `amount`, count duplicate `payment_id` values, and count payments with an amount of zero or less.

In [ ]:
# Your turn

In [ ]:
# Coach answer
payment.select(
    F.count(F.when(F.col("customer_id").isNull(), 1)).alias("null_customer_id"),
    F.count(F.when(F.col("amount").isNull(), 1)).alias("null_amount"),
    F.count(F.when(F.col("amount") <= 0, 1)).alias("non_positive_amount"),
).show()
print("duplicate payment_id:", payment.count() - payment.dropDuplicates(["payment_id"]).count())

### Exercise 3: Orphan checks

Every payment should belong to a known customer, and every customer to a known address. Use left anti joins to count orphans in both relationships.

> *Hint:* `child.join(parent, key, "left_anti")` returns the child rows with no parent.

In [ ]:
# Your turn

In [ ]:
# Coach answer
print("payments without a customer:", payment.join(customer, "customer_id", "left_anti").count())
print("customers without an address:", customer.join(address, "address_id", "left_anti").count())

<a id="customer"></a>
## 2. Revenue per customer

### Exercise 4: Total paid per customer

Create `customer_payments` with one row per `customer_id` and the total amount paid in a column called `total_amount`, rounded to 2 decimals.

In [ ]:
# Your turn

In [ ]:
# Coach answer
customer_payments = (
    payment.groupBy("customer_id")
    .agg(F.round(F.sum("amount"), 2).alias("total_amount"))
)
customer_payments.orderBy(F.desc("total_amount")).show(5)
print("customers who paid:", customer_payments.count())

### Exercise 5: Add customer details

Join `customer_payments` to the customer table, keeping only `customer_id`, `first_name`, `last_name`, `address_id` and `total_amount`. Call the result `customer_payment_details`.

In [ ]:
# Your turn

In [ ]:
# Coach answer
customer_payment_details = customer_payments.join(
    customer.select("customer_id", "first_name", "last_name", "address_id"),
    on="customer_id",
    how="inner",
)
customer_payment_details.show(5)

<a id="joins"></a>
## 3. Join the location tables

### Exercise 6: City and country

Join `city` to `country` to get a table `city_country` with the columns `city_id`, `city` and `country`.

> *Hint:* Select the needed columns from each table first, so `last_update` does not appear twice.

In [ ]:
# Your turn

In [ ]:
# Coach answer
city_country = (
    city.select("city_id", "city", "country_id")
    .join(country.select("country_id", "country"), "country_id", "inner")
    .drop("country_id")
)
city_country.show(5)

### Exercise 7: Address with city and country

Join `address` to `city_country` to get `address_location` with `address_id`, `district`, `city` and `country`.

In [ ]:
# Your turn

In [ ]:
# Coach answer
address_location = (
    address.select("address_id", "district", "city_id")
    .join(city_country, "city_id", "inner")
    .drop("city_id")
)
address_location.show(5)

### Exercise 8: The full table

Join `customer_payment_details` to `address_location` to get `customer_full`: one row per customer with their name, location and total paid. Check that no customer was lost in the joins.

In [ ]:
# Your turn

In [ ]:
# Coach answer
customer_full = customer_payment_details.join(address_location, "address_id", "inner")
customer_full.select("customer_id", "first_name", "last_name", "city", "country", "total_amount").show(5)

print("customers before:", customer_payment_details.count(), "| after:", customer_full.count())

<a id="eu"></a>
## 4. Revenue per country and the EU filter

### Exercise 9: Revenue per country

Create `sales_per_country` with the total revenue and the number of customers per `country`, sorted by revenue (highest first). How many countries are there?

In [ ]:
# Your turn

In [ ]:
# Coach answer
sales_per_country = (
    customer_full.groupBy("country")
    .agg(F.round(F.sum("total_amount"), 2).alias("total_amount"),
         F.count("*").alias("customers"))
    .orderBy(F.desc("total_amount"))
)
sales_per_country.show(10)
print("countries:", sales_per_country.count())

### Exercise 10: EU countries only

Filter `sales_per_country` to the countries in `EU_COUNTRIES`, giving `sales_per_eu_country`. How many EU countries appear in the data, and which ones from the list are missing?

In [ ]:
# Your turn

In [ ]:
# Coach answer
sales_per_eu_country = sales_per_country.filter(F.col("country").isin(EU_COUNTRIES))
sales_per_eu_country.show(30)
print("EU countries with sales:", sales_per_eu_country.count())

present = [r["country"] for r in sales_per_eu_country.select("country").collect()]   # small result: safe to collect
print("EU countries with no customers:", sorted(set(EU_COUNTRIES) - set(present)))

### Exercise 11: Share and rank

Add to `sales_per_eu_country` each country's share of **EU** revenue (in percent, 1 decimal) and its rank. What share of the **total** (worldwide) revenue comes from the EU?

> *Hint:* Compute the totals with `agg(...).first()[0]`, then use them with `F.lit`.

In [ ]:
# Your turn

In [ ]:
# Coach answer
from pyspark.sql.window import Window

eu_total = sales_per_eu_country.agg(F.sum("total_amount")).first()[0]
world_total = sales_per_country.agg(F.sum("total_amount")).first()[0]

ranked_eu = (
    sales_per_eu_country
    .withColumn("share_of_eu_pct", F.round(100 * F.col("total_amount") / F.lit(eu_total), 1))
    .withColumn("rank", F.rank().over(Window.orderBy(F.desc("total_amount"))))
    .orderBy("rank")
)
ranked_eu.show(30)
print(f"EU share of total revenue: {100 * eu_total / world_total:.1f}%")

<a id="sql"></a>
## 5. The same answer in SQL

### Exercise 12: One SQL query

Register the five tables as temporary views and write a **single** SQL query that returns the total revenue per EU country, highest first.

In [ ]:
# Your turn

In [ ]:
# Coach answer
for name, df in [("payment", payment), ("customer", customer), ("address", address), ("city", city), ("country", country)]:
    df.createOrReplaceTempView(name)

eu_list = ", ".join(f"'{c}'" for c in EU_COUNTRIES)
spark.sql(f"""
    SELECT co.country, ROUND(SUM(p.amount), 2) AS total_amount
    FROM payment p
    JOIN customer cu ON p.customer_id = cu.customer_id
    JOIN address  a  ON cu.address_id = a.address_id
    JOIN city     ci ON a.city_id     = ci.city_id
    JOIN country  co ON ci.country_id = co.country_id
    WHERE co.country IN ({eu_list})
    GROUP BY co.country
    ORDER BY total_amount DESC
""").show(30)

<a id="save"></a>
## 6. Visualise and save

### Exercise 13: Bar chart

Bring `sales_per_eu_country` into pandas (it is small) and draw a horizontal bar chart of revenue per EU country, sorted.

In [ ]:
# Your turn

In [ ]:
# Coach answer
import matplotlib.pyplot as plt

pdf = sales_per_eu_country.orderBy("total_amount").toPandas()
pdf.plot.barh(x="country", y="total_amount", figsize=(8, 6), legend=False, color="steelblue")
plt.xlabel("total revenue")
plt.title("Revenue per EU country (Jan to Jul 2022)")
plt.tight_layout()
plt.show()

> **Write permissions.** In a shared training workspace you may not be allowed to write files. The answer cell catches the error and shows the alternative; if neither works, skip this exercise: the rest of the practical does not depend on it.

**Writing output (as in notebook 01.02, Section 18).** A small result is saved as one CSV file **next to this notebook** with pandas: a file name with no folder in front of it. Large results use Spark's writer, which writes a **folder** of part files to a **Unity Catalog Volume** (a path like `/Volumes/<catalog>/<schema>/<volume>/eu_sales`).

### Exercise 14: Save the result

Save `sales_per_eu_country` as `eu_sales.csv` next to this notebook (it is small, so go through pandas), then read the file back with pandas and count the rows. Write, as a comment, the Spark line you would use for a large result.

In [ ]:
# Your turn

In [ ]:
# Coach answer
import pandas as pd

OUTPUT_FILE = "eu_sales.csv"                    # no folder: saved next to this notebook
try:
    sales_per_eu_country.toPandas().to_csv(OUTPUT_FILE, index=False)
    print("rows read back:", len(pd.read_csv(OUTPUT_FILE)))
except OSError as e:
    print("Could not write a file here ->", type(e).__name__, "(training workspaces often do not allow it)")

# For a large result, Spark writes a folder of part files to a Unity Catalog Volume:
# sales_per_eu_country.write.csv("/Volumes/<catalog>/<schema>/<volume>/eu_sales", header=True, mode="overwrite")

**Check your answer.** The expected result is stored in the course bucket as `eu_sales.csv`. If both match, every join and filter above is correct.

In [ ]:
expected = pandas_bridge(spark, "eu_sales")
# Alternative (plain pandas, then Spark): expected = spark.createDataFrame(pd.read_csv(session_datasets_http["eu_sales"]))
# Alternative (Spark reads S3): expected = spark.read.csv(session_datasets["eu_sales"], header=True, inferSchema=True)

diff = (
    sales_per_eu_country.select("country", F.round("total_amount", 2).alias("mine"))
    .join(expected.select("country", F.round("total_amount", 2).alias("expected")), "country", "full")
    .filter(~F.col("mine").eqNullSafe(F.col("expected")))
)
print("rows that differ from the expected result:", diff.count())

<a id="capstone"></a>
## 7. Capstone

### Capstone part 1: Best customer per EU country

For each EU country, find the customer with the highest total paid. Show the country, the customer's full name and their total.

In [ ]:
# Your turn

In [ ]:
# Coach answer
from pyspark.sql.window import Window

w = Window.partitionBy("country").orderBy(F.desc("total_amount"))
(
    customer_full.filter(F.col("country").isin(EU_COUNTRIES))
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select("country", F.concat_ws(" ", "first_name", "last_name").alias("customer"), "total_amount")
    .orderBy(F.desc("total_amount"))
    .show(30, truncate=False)
)

### Capstone part 2: Monthly EU trend

Compute EU revenue per month (`yyyy-MM`) from the payment level, not the customer totals. Which month was strongest?

> *Hint:* A left semi join keeps only the payments of EU customers without adding columns.

In [ ]:
# Your turn

In [ ]:
# Coach answer
eu_customers = customer_full.filter(F.col("country").isin(EU_COUNTRIES)).select("customer_id")

(
    payment.join(eu_customers, "customer_id", "left_semi")
    .withColumn("pay_month", F.date_format(F.to_date(F.substring(F.col("payment_date").cast("string"), 1, 10)), "yyyy-MM"))
    .groupBy("pay_month")
    .agg(F.round(F.sum("amount"), 2).alias("eu_revenue"), F.count("*").alias("payments"))
    .orderBy("pay_month")
    .show()
)

### Capstone part 3: Revenue per customer, EU versus rest of the world

Is the EU small because it has few customers, or because each customer spends less? Compare the number of customers and the average revenue per customer for EU and non-EU countries.

In [ ]:
# Your turn

In [ ]:
# Coach answer
(
    customer_full
    .withColumn("market", F.when(F.col("country").isin(EU_COUNTRIES), "EU").otherwise("Rest of world"))
    .groupBy("market")
    .agg(F.count("*").alias("customers"),
         F.round(F.sum("total_amount"), 2).alias("revenue"),
         F.round(F.avg("total_amount"), 2).alias("avg_per_customer"))
    .show()
)
# Average spend per customer is similar in both markets: the EU's small share is driven by the
# number of customers there, not by lower spending.

<a id="further"></a>
## 8. Further practice

- Repeat the analysis by **district** inside the strongest EU country.
- Use `pivot` to build a country-by-month revenue table for the EU.
- Check the plan of Exercise 8 with `explain()`: which joins did Spark broadcast automatically, and why?
- Rewrite Exercise 11 entirely in SQL with a window function (`RANK() OVER (ORDER BY ...)`).